# [3장 3강] 실습: 결정트리와 앙상블

## 실습 목표

- Decision Tree가 질문을 반복해 고객 이탈을 분류하는 방식을 설명할 수 있다.
- 불순도와 Gini Index의 의미를 설명할 수 있다.
- 기본 트리와 깊이가 제한된 트리의 일반화 성능을 비교할 수 있다.
- Decision Tree 구조를 시각화하고 주요 분기 기준을 해석할 수 있다.
- Ensemble과 Random Forest의 원리를 설명할 수 있다.
- Feature Importance로 주요 고객 이탈 Feature를 확인할 수 있다.

## 사용 데이터

- 파일: `Telco-Customer.csv`
- Label: `Churn` (`No=0`, `Yes=1`)
- Positive Class: 이탈 고객

## 진행 방식

- 문제 설명과 요구사항을 확인한 뒤 코드 셀을 작성합니다.
- 실행 결과를 근거로 문제에 제시된 질문에 답합니다.

## 실습 준비: 데이터 불러오기

Google Colab에서는 파일을 Google Drive의 `MyDrive`에 저장합니다. Jupyter Notebook에서는 노트북과 같은 폴더에 저장합니다.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    file_path='/content/drive/MyDrive/Telco-Customer.csv'
except ModuleNotFoundError:
    file_path='Telco-Customer.csv'

telco_df=pd.read_csv(file_path)
display(telco_df.head())
print('데이터 크기:',telco_df.shape)
telco_df.info()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


데이터 크기: (7043, 21)
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   

## 필수 1: Decision Tree 분류 모델 학습 및 평가

### 문제 1-1: 기본 Decision Tree 모델 학습하기

#### 문제 설명

깊이를 제한하지 않은 Decision Tree로 고객 이탈을 분류합니다. 학습 데이터와 테스트 데이터의 성능 차이를 비교하여 트리가 학습 데이터를 지나치게 세밀하게 분할했는지 확인합니다.

#### 요구사항

1. `customerID`를 제거하고 `TotalCharges`를 숫자로 변환합니다.
2. `Churn`을 `Yes=1`, `No=0`으로 변환합니다.
3. `random_state=42`, `stratify=y`로 학습 80%, 테스트 20%로 분리합니다.
4. 수치형은 중앙값 대체, 범주형은 One-Hot Encoding하는 전처리기를 학습 데이터에 적용합니다.
5. `DecisionTreeClassifier(random_state=42)`를 학습합니다.
6. 학습·테스트 Accuracy, Precision, Recall과 F1-score를 계산합니다.
7. 트리의 깊이와 Leaf Node 개수를 출력합니다.
8. 학습·테스트 성능 차이를 근거로 Overfitting 여부를 설명합니다.

#### 결과 해석 작성

> 기본 Decision Tree의 학습 성능과 테스트 성능 차이는 무엇을 의미하며, 왜 이런 차이가 발생했나요?

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# 1. 데이터 전처리
df = telco_df.copy()

# customerID 제거
df = df.drop(columns=['customerID'])

# TotalCharges 숫자형으로 변환
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Churn 변환: Yes = 1, No = 0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})


# 2. X, y 분리
X = df.drop(columns=['Churn'])
y = df['Churn']


# 3. 학습/테스트 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# 4. 수치형/범주형 변수 구분
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object', 'str']).columns


# 수치형: 중앙값 대체
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

# 범주형: 최빈값 대체 + One-Hot Encoding
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])


# 5. 전처리기 구성
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])


# 6. Decision Tree 모델 구성
dt_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42))
])


# 7. 모델 학습
dt_model.fit(X_train, y_train)


# 8. 예측
y_train_pred = dt_model.predict(X_train)
y_test_pred = dt_model.predict(X_test)


# 9. 평가 함수
def evaluate_model(y_true, y_pred, dataset_name):
    print(f'[{dataset_name}]')
    print(f'Accuracy : {accuracy_score(y_true, y_pred):.4f}')
    print(f'Precision: {precision_score(y_true, y_pred):.4f}')
    print(f'Recall   : {recall_score(y_true, y_pred):.4f}')
    print(f'F1-score : {f1_score(y_true, y_pred):.4f}')
    print()


# 10. 학습/테스트 성능 출력
evaluate_model(y_train, y_train_pred, 'Train')
evaluate_model(y_test, y_test_pred, 'Test')


# 11. 트리 깊이 및 Leaf Node 개수
tree = dt_model.named_steps['classifier']

print('Tree Depth:', tree.get_depth())
print('Leaf Nodes:', tree.get_n_leaves())

[Train]
Accuracy : 0.9980
Precision: 0.9993
Recall   : 0.9933
F1-score : 0.9963

[Test]
Accuracy : 0.7289
Precision: 0.4896
Recall   : 0.5053
F1-score : 0.4974

Tree Depth: 22
Leaf Nodes: 1100


## 필수 2: Decision Tree 구조와 분기 기준 해석

### 문제 2-1: 제한된 Decision Tree 시각화하기

#### 문제 설명

`max_depth=3`으로 모델 복잡도를 제한하고 트리의 상위 분기 조건과 Gini 불순도를 시각적으로 확인합니다.

#### 요구사항

1. `DecisionTreeClassifier(max_depth=3, random_state=42)`를 학습합니다.
2. 학습·테스트 Accuracy, Precision, Recall과 F1-score를 계산합니다.
3. `plot_tree()`로 전체 트리를 시각화합니다.
4. 노드의 Feature, 분할 기준, Gini, Samples, Value와 Class를 확인합니다.
5. Root Node Feature와 기준값을 코드로 출력합니다.
6. 기본 트리와 제한된 트리의 테스트 성능을 비교합니다.
7. Gini가 낮다는 의미를 설명합니다.

#### 결과 해석 작성

> Root Node는 어떤 Feature와 기준으로 고객을 나누며, Gini가 낮아진다는 것은 무엇을 의미하나요?

## 필수 3: Random Forest 학습 및 성능 비교

### 문제 3-1: Random Forest 분류 모델 학습하기

#### 문제 설명

서로 다른 데이터와 Feature를 사용하는 여러 Decision Tree의 다수결로 고객 이탈을 예측하고 단일 트리와 성능을 비교합니다.

#### 요구사항

1. `RandomForestClassifier(n_estimators=100, random_state=42)`를 학습합니다.
2. 학습·테스트 Accuracy, Precision, Recall과 F1-score를 계산합니다.
3. 기본 Decision Tree와 Random Forest의 테스트 성능 비교표를 만듭니다.
4. 두 모델의 학습·테스트 성능 차이를 비교합니다.
5. Random Forest가 단일 Decision Tree보다 일반적으로 Overfitting에 강한 이유를 설명합니다.
6. 모든 평가지표가 반드시 좋아졌는지 실제 결과로 확인합니다.

#### 결과 해석 작성

> Random Forest는 기본 Decision Tree보다 어떤 테스트 지표가 좋아졌고 어떤 지표는 조금 낮아졌나요? 여러 트리를 결합하면 왜 일반적으로 예측이 안정적인가요?

## 심화 1: Random Forest 내부 트리와 Feature Importance

### 문제 4-1: 내부 Decision Tree와 주요 Feature 시각화하기

#### 문제 설명

Random Forest 내부의 첫 번째 Decision Tree를 확인하고, 전체 Forest가 불순도 감소에 많이 사용한 Feature를 찾습니다.

#### 요구사항

1. `estimators_[0]`으로 첫 번째 내부 트리를 선택합니다.
2. `max_depth=3`을 지정해 내부 트리의 상위 3단계만 시각화합니다.
3. `feature_importances_`를 Feature 이름과 결합합니다.
4. 중요도 내림차순으로 정렬하고 상위 10개를 출력합니다.
5. 상위 10개를 가로 막대그래프로 시각화합니다.
6. 가장 중요한 Feature를 출력합니다.
7. Feature Importance가 높다는 의미와 해석 시 주의점을 설명합니다.

#### 결과 해석 작성

> 가장 높은 Feature Importance를 가진 Feature는 무엇이며, Importance가 높다는 사실을 고객 이탈의 직접적인 원인으로 해석해도 되나요?

## 실습 마무리

1. 분류 트리와 회귀 트리는 Leaf Node에서 무엇을 예측하나요?
2. 트리 깊이를 제한하면 Bias와 Variance는 일반적으로 어떻게 변하나요?
3. Random Forest가 다양한 트리를 만드는 두 가지 방법은 무엇인가요?